# Лабораторная работа 3. Сегментация дорожных знаков

**Цель:** обучить модель YOLOv11-seg сегментировать 8 типов российских дорожных знаков.

**Содержание:**
1. Загрузка и подготовка датасета Russian Road Signs Segmentation
2. Обучение YOLOv11n-seg (fine-tune)
3. Оценка качества: IoU, Precision, Recall, L2
4. Инференс на видео с дорожными знаками
5. Трекинг: ByteTrack и BoT-SORT, метрика ID Switches

In [ ]:
import os, sys, glob, json, shutil, gc, random
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

BASE_DIR = os.path.dirname(os.path.abspath('__file__')) if '__file__' not in dir() else os.path.dirname(os.path.abspath(__file__))
YOLO_DIR = os.path.join(BASE_DIR, 'yolo_dataset')
DATASET_ROOT = os.path.join(BASE_DIR, 'data', 'sign_dataset')
VIDEO_DIR = os.path.join(BASE_DIR, 'videos')

BEST_MODEL_PATH = os.path.join(BASE_DIR, 'best.pt')
if not os.path.exists(BEST_MODEL_PATH):
    candidates = glob.glob(os.path.join(BASE_DIR, 'runs', 'segment', 'road_signs_seg*', 'weights', 'best.pt'))
    if candidates:
        BEST_MODEL_PATH = candidates[-1]

os.makedirs(VIDEO_DIR, exist_ok=True)
print(f"BASE_DIR: {BASE_DIR}")
print(f"BEST_MODEL_PATH: {BEST_MODEL_PATH} (exists: {os.path.exists(BEST_MODEL_PATH)})")
print(f"VIDEO_DIR: {VIDEO_DIR}")

## 1. Загрузка датасета

Датасет [Russian Road Signs Segmentation](https://www.kaggle.com/datasets/viacheslavshalamov/russian-road-signs-segmentation-dataset) содержит ~100 000 изображений дорожных знаков России с масками сегментации для 8 типов знаков.

In [ ]:
if os.path.exists(DATASET_ROOT):
    train_files = os.listdir(os.path.join(DATASET_ROOT, 'train'))
    val_files = os.listdir(os.path.join(DATASET_ROOT, 'val'))
    train_imgs = [f for f in train_files if f.endswith('.jpg')]
    val_imgs = [f for f in val_files if f.endswith('.jpg')]
    print(f"Датасет: {DATASET_ROOT}")
    print(f"  Train: {len(train_imgs)} images, {len(train_files) - len(train_imgs)} json")
    print(f"  Val:   {len(val_imgs)} images, {len(val_files) - len(val_imgs)} json")
else:
    print(f"Датасет не найден в {DATASET_ROOT}")
    print("Скачайте его или запустите train.py")

In [ ]:
sample_json = os.path.join(DATASET_ROOT, 'train', '1.jpg_coco.json')
if os.path.exists(sample_json):
    with open(sample_json) as f:
        ann = json.load(f)
    print(f"Пример аннотации (1.jpg_coco.json):")
    print(f"  Ключи: {list(ann.keys())}")
    print(f"  class_ids: {ann.get('class_ids', [])}")
    print(f"  masks shape: {np.array(ann.get('masks', [])).shape}")
    print(f"  rois: {ann.get('rois', [])[:2]}")
    print(f"  scores: {[round(s,3) for s in ann.get('scores', [])[:3]]}")


In [ ]:
all_images = sorted(glob.glob(os.path.join(DATASET_ROOT, 'train', '*.jpg')))
sample_imgs = all_images[::max(1, len(all_images) // 8)][:8]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i < len(sample_imgs):
        img = cv2.imread(sample_imgs[i])
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            ax.set_title(os.path.basename(sample_imgs[i])[:25], fontsize=8)
    ax.axis('off')
plt.suptitle('Примеры изображений из датасета', fontsize=14)
plt.tight_layout()
plt.show()


## 2. Конвертация в формат YOLO-seg

Конвертируем аннотации датасета в формат YOLOv11 instance segmentation.
Каждый файл `.txt` содержит строки: `class_id x1 y1 x2 y2 ... xn yn` (нормализованные координаты полигона).

Автоматически определяем формат датасета и конвертируем в нужный формат.

In [ ]:
SEED = 42

SIGN_CLASSES = {
    1: 'prohibitory',    # Запрещающие
    2: 'mandatory',      # Предписывающие
    3: 'warning',        # Предупреждающие
    4: 'priority',       # Приоритета
    5: 'informational',  # Информационные
    6: 'special',        # Особых предписаний
    7: 'service',        # Сервиса
    8: 'additional',     # Дополнительной информации
}

CLASS_NAMES = list(SIGN_CLASSES.values())
CLASS_ID_TO_IDX = {cid: i for i, cid in enumerate(SIGN_CLASSES.keys())}

def mask_roi_to_polygon(small_mask, roi, img_h, img_w):
    """
    Convert a small boolean mask (56x56) within an ROI to a normalized polygon.
    roi = [y1, x1, y2, x2] in pixel coordinates.
    Returns list of normalized [x, y, x, y, ...] or None.
    """
    y1, x1, y2, x2 = roi
    roi_h = max(y2 - y1, 1)
    roi_w = max(x2 - x1, 1)

    mask_uint8 = (np.array(small_mask) > 0).astype(np.uint8) * 255
    mask_resized = cv2.resize(mask_uint8, (roi_w, roi_h), interpolation=cv2.INTER_NEAREST)

    contours, _ = cv2.findContours(mask_resized, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    contour = max(contours, key=cv2.contourArea)
    if cv2.contourArea(contour) < 4:
        return None
    epsilon = 0.01 * cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, epsilon, True)
    if len(approx) < 3:
        return None

    pts = approx.reshape(-1, 2).astype(float)
    pts[:, 0] += x1
    pts[:, 1] += y1

    normalized = []
    for px, py in pts:
        nx = max(0.0, min(1.0, round(px / img_w, 6)))
        ny = max(0.0, min(1.0, round(py / img_h, 6)))
        normalized.extend([nx, ny])
    return normalized


def convert_split(split_path, out_dir):
    """Convert one split (train or val) to YOLO-seg format."""
    os.makedirs(os.path.join(out_dir, 'images'), exist_ok=True)
    os.makedirs(os.path.join(out_dir, 'labels'), exist_ok=True)

    all_files = sorted(os.listdir(split_path))
    img_files = [f for f in all_files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    print(f"  Изображений: {len(img_files)}")

    count = 0
    skipped = 0

    for img_name in img_files:
        json_path = os.path.join(split_path, img_name + '_coco.json')
        if not os.path.exists(json_path):
            skipped += 1
            continue

        with open(json_path, 'r') as f:
            ann = json.load(f)

        masks_3d = np.array(ann.get('masks', []))     # (H_mask, W_mask, N)
        class_ids = ann.get('class_ids', [])            # (N,)
        rois = ann.get('rois', [])                      # (N, 4) — [y1, x1, y2, x2]

        n_objects = len(class_ids)
        if n_objects == 0:
            skipped += 1
            continue

        img_data = cv2.imread(os.path.join(split_path, img_name))
        if img_data is None:
            skipped += 1
            continue
        img_h, img_w = img_data.shape[:2]

        label_lines = []
        for i in range(n_objects):
            cls_id = class_ids[i]
            cls_idx = CLASS_ID_TO_IDX.get(cls_id, None)
            if cls_idx is None:
                cls_idx = max(0, min(cls_id - 1, len(CLASS_NAMES) - 1))

            roi = rois[i]

            if masks_3d.ndim == 3:
                small_mask = masks_3d[:, :, i]
            else:
                continue

            poly = mask_roi_to_polygon(small_mask, roi, img_h, img_w)
            if poly and len(poly) >= 6:
                coords = ' '.join(f'{c:.6f}' for c in poly)
                label_lines.append(f'{cls_idx} {coords}')

        if label_lines:
            shutil.copy2(
                os.path.join(split_path, img_name),
                os.path.join(out_dir, 'images', img_name)
            )
            stem = Path(img_name).stem
            with open(os.path.join(out_dir, 'labels', stem + '.txt'), 'w') as f:
                f.write('\n'.join(label_lines))
            count += 1

    if skipped > 0:
        print(f"  Пропущено: {skipped}")
    return count


# --------------- Convert train and val ---------------

print("Конвертация масок → YOLO-seg полигоны...\n")

if os.path.exists(YOLO_DIR):
    shutil.rmtree(YOLO_DIR)

for split_name in ['train', 'val']:
    split_path = os.path.join(DATASET_ROOT, split_name)
    out_dir = os.path.join(YOLO_DIR, split_name)
    print(f"--- {split_name} ---")
    c = convert_split(split_path, out_dir)
    print(f"  Конвертировано: {c}\n")

# --------------- data.yaml ---------------

train_count = len(glob.glob(os.path.join(YOLO_DIR, 'train', 'images', '*.*')))
val_count = len(glob.glob(os.path.join(YOLO_DIR, 'val', 'images', '*.*')))
print(f"Итого: Train={train_count}, Val={val_count}")

yaml_content = f"""path: {YOLO_DIR}
train: train/images
val: val/images

nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
"""
yaml_path = os.path.join(YOLO_DIR, 'data.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"\ndata.yaml:")
print(yaml_content)

# Проверка: показать пример label
sample_labels = glob.glob(os.path.join(YOLO_DIR, 'train', 'labels', '*.txt'))
if sample_labels:
    with open(sample_labels[0]) as f:
        print(f"Пример label ({os.path.basename(sample_labels[0])}):")
        print(f.read()[:300])


## 3. Обучение YOLOv11n-seg

Дообучаем предобученную модель YOLOv11 nano для instance segmentation на нашем датасете.
Используем Transfer Learning: веса COCO адаптируются под 8 классов дорожных знаков.

In [ ]:
if os.path.exists(BEST_MODEL_PATH):
    print(f"Обученная модель найдена: {BEST_MODEL_PATH}")
    print(f"Размер: {os.path.getsize(BEST_MODEL_PATH) / 1e6:.1f} MB")
    print("Обучение пропущено. Для переобучения удалите best.pt и перезапустите.")
else:
    print("best.pt не найден. Запускаем обучение...")
    print("(Или запустите в терминале: python3 train.py)")
    model = YOLO('yolo11n-seg.pt')
    model.train(
        data=os.path.join(YOLO_DIR, 'data.yaml'),
        epochs=50,
        imgsz=640,
        batch=16,
        patience=10,
        name='road_signs_seg',
        save=True,
        plots=True,
        workers=4,
    )
    trained = glob.glob(os.path.join('runs', 'segment', 'road_signs_seg*', 'weights', 'best.pt'))
    if trained:
        shutil.copy2(trained[-1], os.path.join(BASE_DIR, 'best.pt'))
        BEST_MODEL_PATH = os.path.join(BASE_DIR, 'best.pt')
        print(f"\nbest.pt сохранён: {BEST_MODEL_PATH}")


In [ ]:
from IPython.display import Image, display

train_dir = 'runs/segment/road_signs_seg'
for plot_name in ['results.png', 'confusion_matrix.png', 'P_curve.png', 'R_curve.png']:
    plot_path = os.path.join(train_dir, plot_name)
    if os.path.exists(plot_path):
        print(f"--- {plot_name} ---")
        display(Image(filename=plot_path, width=800))


## 4. Оценка качества сегментации

Оцениваем качество на валидационной выборке:
- Стандартные YOLO-метрики (mAP, Precision, Recall)
- Попиксельный IoU для каждого изображения
- L2-расстояние между контурами масок
- Процент изображений с IoU ≥ 0.5, ≥ 0.75, ≥ 0.9

In [ ]:
best_model = YOLO(BEST_MODEL_PATH)

metrics = best_model.val(
    data=os.path.join(YOLO_DIR, 'data.yaml'),
    split='val',
)

print("\n=== Стандартные YOLO-метрики ===")
print(f"Mask  Precision: {metrics.seg.mp:.4f}")
print(f"Mask  Recall:    {metrics.seg.mr:.4f}")
print(f"Mask  mAP@50:    {metrics.seg.map50:.4f}")
print(f"Mask  mAP@50-95: {metrics.seg.map:.4f}")
print(f"Box   Precision: {metrics.box.mp:.4f}")
print(f"Box   Recall:    {metrics.box.mr:.4f}")
print(f"Box   mAP@50:    {metrics.box.map50:.4f}")


In [ ]:
from scipy.ndimage import distance_transform_edt

def get_boundary(mask, thickness=1):
    """Extract boundary pixels from a binary mask."""
    kernel = np.ones((2 * thickness + 1, 2 * thickness + 1), np.uint8)
    eroded = cv2.erode(mask.astype(np.uint8), kernel)
    return (mask.astype(np.uint8) - eroded).astype(bool)

def compute_pixel_metrics(pred_mask, gt_mask):
    """Compute IoU, Precision, Recall, L2 between two binary masks."""
    pred = pred_mask.astype(bool)
    gt = gt_mask.astype(bool)

    intersection = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    iou = float(intersection) / float(union) if union > 0 else 0.0

    tp = intersection
    fp = np.logical_and(pred, ~gt).sum()
    fn = np.logical_and(~pred, gt).sum()
    precision = float(tp) / float(tp + fp) if (tp + fp) > 0 else 0.0
    recall = float(tp) / float(tp + fn) if (tp + fn) > 0 else 0.0

    pred_boundary = get_boundary(pred)
    gt_boundary = get_boundary(gt)
    if pred_boundary.sum() > 0 and gt_boundary.sum() > 0:
        dt_gt = distance_transform_edt(~gt_boundary)
        dt_pred = distance_transform_edt(~pred_boundary)
        l2 = (dt_gt[pred_boundary].mean() + dt_pred[gt_boundary].mean()) / 2.0
    else:
        l2 = float('inf')

    return {'iou': iou, 'precision': precision, 'recall': recall, 'l2': l2}


def polygon_to_mask(polygon, img_w, img_h):
    """Convert YOLO normalized polygon to binary mask."""
    pts = np.array(polygon).reshape(-1, 2)
    pts[:, 0] *= img_w
    pts[:, 1] *= img_h
    pts = pts.astype(np.int32)
    mask = np.zeros((img_h, img_w), dtype=np.uint8)
    cv2.fillPoly(mask, [pts], 255)
    return mask


def parse_yolo_label(label_path):
    """Parse YOLO segmentation label file -> list of (class_id, polygon_coords)."""
    entries = []
    if not os.path.exists(label_path):
        return entries
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 7:
                cls_id = int(parts[0])
                coords = [float(x) for x in parts[1:]]
                entries.append((cls_id, coords))
    return entries


# Run evaluation on validation set
val_img_dir = os.path.join(YOLO_DIR, 'val', 'images')
val_lbl_dir = os.path.join(YOLO_DIR, 'val', 'labels')

val_images = sorted(glob.glob(os.path.join(val_img_dir, '*.*')))
print(f"Валидация на {len(val_images)} изображениях...")

all_metrics = []
for img_path in val_images:
    img = cv2.imread(img_path)
    if img is None:
        continue
    h, w = img.shape[:2]
    stem = Path(img_path).stem

    gt_entries = parse_yolo_label(os.path.join(val_lbl_dir, stem + '.txt'))
    if not gt_entries:
        continue

    gt_mask = np.zeros((h, w), dtype=np.uint8)
    for cls_id, coords in gt_entries:
        gt_mask = np.maximum(gt_mask, polygon_to_mask(coords, w, h))

    results = best_model.predict(img_path, verbose=False)
    pred_mask = np.zeros((h, w), dtype=np.uint8)
    if results and results[0].masks is not None:
        for seg_mask in results[0].masks.data:
            m = seg_mask.cpu().numpy()
            m_resized = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
            pred_mask = np.maximum(pred_mask, (m_resized > 0.5).astype(np.uint8) * 255)

    m = compute_pixel_metrics(pred_mask, gt_mask)
    m['image'] = os.path.basename(img_path)
    all_metrics.append(m)

    if len(all_metrics) % 500 == 0:
        print(f"  Обработано {len(all_metrics)}/{len(val_images)} ...")

print(f"Оценено: {len(all_metrics)} изображений")


In [ ]:
import pandas as pd

if all_metrics:
    df = pd.DataFrame(all_metrics)

    mean_iou = df['iou'].mean()
    mean_prec = df['precision'].mean()
    mean_rec = df['recall'].mean()
    finite_l2 = df['l2'][df['l2'] != float('inf')]
    mean_l2 = finite_l2.mean() if len(finite_l2) > 0 else float('inf')

    pct_05 = (df['iou'] >= 0.5).mean() * 100
    pct_075 = (df['iou'] >= 0.75).mean() * 100
    pct_09 = (df['iou'] >= 0.9).mean() * 100

    print("=" * 55)
    print("       РЕЗУЛЬТАТЫ ОЦЕНКИ НА ВАЛИДАЦИИ")
    print("=" * 55)
    print(f"{'Метрика':<25} {'Значение':>10}")
    print("-" * 55)
    print(f"{'Mean IoU':<25} {mean_iou:>10.4f}")
    print(f"{'Mean Precision':<25} {mean_prec:>10.4f}")
    print(f"{'Mean Recall':<25} {mean_rec:>10.4f}")
    print(f"{'Mean L2 (boundary)':<25} {mean_l2:>10.4f}")
    print("-" * 55)
    print(f"{'% IoU >= 0.50':<25} {pct_05:>9.1f}%")
    print(f"{'% IoU >= 0.75':<25} {pct_075:>9.1f}%")
    print(f"{'% IoU >= 0.90':<25} {pct_09:>9.1f}%")
    print("=" * 55)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].hist(df['iou'], bins=50, edgecolor='black', alpha=0.7)
    axes[0].axvline(0.5, color='r', linestyle='--', label='IoU=0.5')
    axes[0].axvline(0.75, color='orange', linestyle='--', label='IoU=0.75')
    axes[0].axvline(0.9, color='g', linestyle='--', label='IoU=0.9')
    axes[0].set_xlabel('IoU')
    axes[0].set_ylabel('Количество')
    axes[0].set_title('Распределение IoU')
    axes[0].legend()

    axes[1].hist(df['precision'], bins=50, edgecolor='black', alpha=0.7, color='green')
    axes[1].set_xlabel('Precision')
    axes[1].set_title('Распределение Precision')

    axes[2].hist(df['recall'], bins=50, edgecolor='black', alpha=0.7, color='orange')
    axes[2].set_xlabel('Recall')
    axes[2].set_title('Распределение Recall')

    plt.suptitle('Метрики сегментации на валидации', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("Нет результатов для отображения.")


## 5. Инференс на видео

Запускаем обученную модель на видео с дорожными знаками.

**Загрузите ваши видео** (3+ штуки, по 30+ секунд каждое) в папку `videos/`. Видео должны содержать дорожные знаки, снятые около Университета ИТМО или в других узнаваемых местах.

In [ ]:
OUTPUT_DIR = os.path.join(BASE_DIR, 'video_results')
os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

video_files = sorted([
    f for f in os.listdir(VIDEO_DIR)
    if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))
])

if not video_files:
    print(f"Папка {VIDEO_DIR} пуста!")
    print("Положите туда 3+ видео (30+ сек) с дорожными знаками и перезапустите ячейку.")
else:
    print(f"Видео для обработки ({len(video_files)}):")
    for vf in video_files:
        vp = os.path.join(VIDEO_DIR, vf)
        cap = cv2.VideoCapture(vp)
        fps = cap.get(cv2.CAP_PROP_FPS)
        frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        dur = frames / fps if fps > 0 else 0
        cap.release()
        print(f"  {vf}: {dur:.0f}s, {frames} frames, {fps:.0f} fps")

    seg_model = YOLO(BEST_MODEL_PATH)

    for vf in video_files:
        video_path = os.path.join(VIDEO_DIR, vf)
        print(f"\nОбработка {vf}...")

        results = seg_model.predict(
            source=video_path,
            save=True,
            stream=True,
            conf=0.25,
            project=OUTPUT_DIR,
            name=Path(vf).stem,
            show_labels=True,
            show_conf=True,
            vid_stride=2,
        )
        frame_count = sum(1 for _ in results)
        print(f"  Обработано кадров: {frame_count}")
        gc.collect()

    del seg_model
    gc.collect()

    output_videos = glob.glob(os.path.join(OUTPUT_DIR, '**', '*.avi'), recursive=True)
    output_videos += glob.glob(os.path.join(OUTPUT_DIR, '**', '*.mp4'), recursive=True)
    print(f"\nСохранённые видео ({len(output_videos)}):")
    for ov in output_videos:
        print(f"  {ov}")


## 6. Трекинг объектов

Запускаем два алгоритма трекинга — **ByteTrack** и **BoT-SORT** — на отснятых видео.
Оцениваем результат с помощью метрики **ID Switches** (количество смен идентификатора для одного и того же объекта).

In [ ]:
def run_tracking(video_path, tracker_yaml, save_name):
    """Run tracking and return per-frame track data."""
    trk_model = YOLO(BEST_MODEL_PATH)
    results = trk_model.track(
        source=video_path,
        tracker=tracker_yaml,
        save=True,
        stream=True,
        conf=0.25,
        project=os.path.join(BASE_DIR, 'tracking_results'),
        name=save_name,
    )
    frame_tracks = []
    for r in results:
        tracks = {}
        if r.boxes is not None and r.boxes.id is not None:
            for box, tid in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.id.cpu().numpy().astype(int)):
                tracks[int(tid)] = box.tolist()
        frame_tracks.append(tracks)
    del trk_model
    gc.collect()
    return frame_tracks


def compute_box_iou(box1, box2):
    """IoU between two [x1,y1,x2,y2] boxes."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0


def count_id_switches(frame_tracks, iou_thresh=0.3):
    """
    Count ID switches: when the same physical object gets a different track ID.
    Match objects between consecutive frames by IoU.
    """
    switches = 0
    prev_tracks = {}

    for curr_tracks in frame_tracks:
        if not prev_tracks or not curr_tracks:
            prev_tracks = curr_tracks
            continue

        prev_ids = list(prev_tracks.keys())
        curr_ids = list(curr_tracks.keys())

        matched_prev = set()
        matched_curr = set()

        pairs = []
        for pid in prev_ids:
            for cid in curr_ids:
                iou = compute_box_iou(prev_tracks[pid], curr_tracks[cid])
                if iou > iou_thresh:
                    pairs.append((iou, pid, cid))
        pairs.sort(reverse=True)

        for iou, pid, cid in pairs:
            if pid in matched_prev or cid in matched_curr:
                continue
            matched_prev.add(pid)
            matched_curr.add(cid)
            if pid != cid:
                switches += 1

        prev_tracks = curr_tracks

    return switches

all_tracking_results = {}

for vf in video_files:
    video_path = os.path.join(VIDEO_DIR, vf)
    print(f"\n{'='*50}")
    print(f"Видео: {vf}")
    print(f"{'='*50}")

    print("\n--- ByteTrack ---")
    bt_tracks = run_tracking(video_path, 'bytetrack.yaml', f'{Path(vf).stem}_bytetrack')
    bt_switches = count_id_switches(bt_tracks)
    bt_unique_ids = len(set(tid for ft in bt_tracks for tid in ft.keys()))
    print(f"  Уникальных треков: {bt_unique_ids}")
    print(f"  ID Switches: {bt_switches}")

    print("\n--- BoT-SORT ---")
    bs_tracks = run_tracking(video_path, 'botsort.yaml', f'{Path(vf).stem}_botsort')
    bs_switches = count_id_switches(bs_tracks)
    bs_unique_ids = len(set(tid for ft in bs_tracks for tid in ft.keys()))
    print(f"  Уникальных треков: {bs_unique_ids}")
    print(f"  ID Switches: {bs_switches}")

    all_tracking_results[vf] = {
        'bytetrack': {'id_switches': bt_switches, 'unique_ids': bt_unique_ids, 'frames': len(bt_tracks)},
        'botsort': {'id_switches': bs_switches, 'unique_ids': bs_unique_ids, 'frames': len(bs_tracks)},
    }


In [ ]:
print("\n" + "=" * 70)
print("     СРАВНЕНИЕ ТРЕКЕРОВ")
print("=" * 70)
print(f"{'Видео':<25} {'Трекер':<12} {'ID Switches':>12} {'Треков':>8} {'Кадров':>8}")
print("-" * 70)

for vf, res in all_tracking_results.items():
    name = Path(vf).stem[:22]
    bt = res['bytetrack']
    bs = res['botsort']
    print(f"{name:<25} {'ByteTrack':<12} {bt['id_switches']:>12} {bt['unique_ids']:>8} {bt['frames']:>8}")
    print(f"{'':<25} {'BoT-SORT':<12} {bs['id_switches']:>12} {bs['unique_ids']:>8} {bs['frames']:>8}")
    print("-" * 70)

print("=" * 70)

total_bt = sum(r['bytetrack']['id_switches'] for r in all_tracking_results.values())
total_bs = sum(r['botsort']['id_switches'] for r in all_tracking_results.values())
print(f"\nВсего ID Switches:")
print(f"  ByteTrack: {total_bt}")
print(f"  BoT-SORT:  {total_bs}")

if total_bt < total_bs:
    print(f"\nByteTrack показал меньше ID Switches ({total_bt} vs {total_bs})")
elif total_bs < total_bt:
    print(f"\nBoT-SORT показал меньше ID Switches ({total_bs} vs {total_bt})")
else:
    print(f"\nОба трекера показали одинаковое количество ID Switches ({total_bt})")


## 7. Выводы

### Сегментация
1. **Модель:** YOLOv11n-seg, дообученная на датасете Russian Road Signs Segmentation (~100 000 изображений, 8 классов).
2. **Метрики на валидации:** IoU, Precision, Recall и L2 рассчитаны попиксельно для каждого изображения.
3. **Пороги IoU:** рассчитан процент изображений с IoU ≥ 0.5, IoU ≥ 0.75 и IoU ≥ 0.9.

### Трекинг
4. **Два алгоритма:** ByteTrack и BoT-SORT запущены на отснятых видео.
5. **ID Switches:** подсчитаны для каждого трекера и видео — метрика показывает стабильность трекинга.

### Инференс
6. Модель протестирована на видео с улиц с дорожными знаками, результаты сегментации визуализированы.